# Intent Algebra — Steps 1–2 calibration audit

This notebook is a **read-only view** of a pipeline run. It does not call a model, parse code, choose labels, or rewrite artifacts. Every table below comes from versioned JSON/JSONL files in one run directory, so a result can be traced from message → verb+noun claim → candidate queue → every threshold-accepted single-sided AST subtree → review decision.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display

def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'experiments' / 'intent-algebra' / 'audit_notebook.py').exists():
            return path
    raise RuntimeError('Could not locate the repository root')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / 'experiments' / 'intent-algebra'))

from audit_notebook import (
    artifact_inventory, calibration_funnel, diff_html, load_run,
    outcome_counts, queue_for_claim, record_bundle, subtree_html,
    verify_artifact_index,
)

RUN_ROOT = Path(os.environ.get(
    'INTENT_ALGEBRA_RUN_ROOT',
    REPO_ROOT / '.data' / 'intent-message-code-calibration-1000',
)).resolve()
RUN_ROOT

## 1. Artifact integrity and provenance
The persisted index contains the expected hash for each artifact. A mismatch means the run changed after it was indexed and must not be treated as reproducible. Missing files are shown as pending so a partially completed run remains inspectable.

In [ ]:
integrity = pd.DataFrame(verify_artifact_index(RUN_ROOT))
display(integrity)
if not integrity.empty and integrity['status'].eq('hash_mismatch').any():
    display(Markdown('**STOP:** at least one artifact hash does not match the run index.'))

In [ ]:
run = load_run(RUN_ROOT)
manifest = run['manifest']
if manifest:
    display(pd.json_normalize(manifest, sep='.').T.rename(columns={0: 'value'}))
else:
    display(Markdown('Run manifest is **pending**.'))

## 2. Calibration funnel
This distinguishes claims from emitted datapoints. One claim can emit two linked datapoints when it is independently supported by one `before` subtree and one `after` subtree.

In [ ]:
funnel = pd.DataFrame(calibration_funnel(run))
display(funnel)
if funnel['count'].sum() > 0:
    ax = funnel.plot.barh(x='stage', y='count', legend=False, figsize=(8, 3.5), color='#3b82f6')
    ax.set_title('Steps 1–2 calibration funnel')
    ax.set_xlabel('Count')
    plt.tight_layout()

In [ ]:
reasons = pd.DataFrame(
    [{'reason': reason, 'count': count} for reason, count in outcome_counts(run).items()]
).sort_values('count', ascending=False) if run['unresolved'] else pd.DataFrame(columns=['reason', 'count'])
display(reasons if not reasons.empty else Markdown('No unresolved-claim artifact yet.'))

## 3. Trace one record end to end
Change `RECORD_ID` and `CLAIM_ID` to inspect another item. The queue table is the complete deduplicated evaluation order—not just the winner.

In [ ]:
RECORD_ID = run['records'][0]['record_id'] if run['records'] else None
bundle = record_bundle(run, RECORD_ID) if RECORD_ID else None
if bundle:
    record = bundle['record']
    display(Markdown(f"### Record `{RECORD_ID}`\n\n**Message:** {record.get('message', '')}"))
    display(HTML(diff_html(record.get('before_source', ''), record.get('after_source', ''))))
    display(pd.DataFrame(bundle['claims']))
else:
    display(Markdown('Record-level evidence is **pending**.'))

In [ ]:
CLAIM_ID = bundle['claims'][0]['claim_id'] if bundle and bundle['claims'] else None
queue = queue_for_claim(run, CLAIM_ID) if CLAIM_ID else []
display(Markdown(f"### Candidate queue for `{CLAIM_ID}`") if CLAIM_ID else Markdown('Claim queue is **pending**.'))
if queue:
    preferred = [
        'queue_rank', 'candidate_id', 'side', 'queue_sign', 'candidate_tier',
        'node_type', 'start_byte', 'end_byte', 'alignment_confidence', 'decision',
    ]
    queue_df = pd.DataFrame(queue)
    display(queue_df[[column for column in preferred if column in queue_df.columns]])
else:
    display(Markdown('No AST candidates recorded yet.'))

In [ ]:
selected = [row for row in run['labels'] if row.get('claim_id') == CLAIM_ID]
if selected:
    display(Markdown('### Selected single-sided AST subtree datapoint(s)'))
    for label in selected:
        display(HTML(subtree_html(label)))
        display(pd.DataFrame([label]).T.rename(columns={0: 'value'}))
else:
    display(Markdown('No subtree was selected for this claim. Check `unresolved.jsonl` for the reason.'))

## 4. Human-review audit
The review sample is selected deterministically and review decisions are append-only. Review never overwrites model output; it adds a linked judgment.

In [ ]:
review_sample = pd.DataFrame(run['review_sample'])
review_decisions = pd.DataFrame(run['review_decisions'])
display(Markdown(f"Sampled: **{len(review_sample)}** · reviewed: **{len(review_decisions)}**"))
if not review_decisions.empty:
    decision_column = next((c for c in ['judgment', 'decision', 'label'] if c in review_decisions), None)
    display(review_decisions[decision_column].value_counts(dropna=False).rename('count').to_frame() if decision_column else review_decisions)
else:
    display(Markdown('Human review decisions are **pending**.'))

## 5. Calibration slices and go/no-go evidence
The executor records the pre-registered threshold values and measured metrics in `calibration-summary.json`. This notebook displays them; it does not decide or mutate the run.

In [ ]:
summary = run['summary']
if summary:
    display(pd.json_normalize(summary, sep='.').T.rename(columns={0: 'value'}))
else:
    display(Markdown('Calibration summary is **pending**.'))

if run['labels']:
    labels_df = pd.DataFrame(run['labels'])
    for column in ['language', 'side', 'queue_sign', 'node_type', 'confidence_band']:
        if column in labels_df:
            display(labels_df[column].value_counts(dropna=False).rename('count').to_frame().style.set_caption(column))